In [2]:
import json
import torch
from pathlib import Path

processed_dir = Path("../data/processed")

with open(processed_dir / "vocabulary.json", "r", encoding="utf-8") as f:
    token_to_id = json.load(f)

train_data = torch.load(processed_dir / "train_tensors.pt")
val_data = torch.load(processed_dir / "val_tensors.pt")
test_data = torch.load(processed_dir / "test_tensors.pt")

print("Vocabulary size:", len(token_to_id))
print("Train source shape:", train_data["source"].shape)
print("Train decoder input shape:", train_data["decoder_input"].shape)
print("Train decoder target shape:", train_data["decoder_target"].shape)

Vocabulary size: 9537
Train source shape: torch.Size([800, 512])
Train decoder input shape: torch.Size([800, 127])
Train decoder target shape: torch.Size([800, 127])


In [3]:
PAD_ID = token_to_id["<PAD>"]
UNK_ID = token_to_id["<UNK>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]
SEP_ID = token_to_id["<SEP>"]

print("PAD ID:", PAD_ID)
print("UNK ID:", UNK_ID)
print("BOS ID:", BOS_ID)
print("EOS ID:", EOS_ID)
print("SEP ID:", SEP_ID)

PAD ID: 0
UNK ID: 1
BOS ID: 2
EOS ID: 3
SEP ID: 4


In [4]:
#Transformer hyperparameters
VOCAB_SIZE = len(token_to_id)

D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2

D_FF = 512
DROPOUT = 0.1

print("Vocabulary size:", VOCAB_SIZE)
print("Model dimension:", D_MODEL)
print("Number of attention heads:", NUM_HEADS)
print("Encoder layers:", NUM_ENCODER_LAYERS)
print("Decoder layers:", NUM_DECODER_LAYERS)
print("Feed-forward dimension:", D_FF)
print("Dropout:", DROPOUT)

print("\nDimension per attention head:", D_MODEL // NUM_HEADS)

Vocabulary size: 9537
Model dimension: 128
Number of attention heads: 4
Encoder layers: 2
Decoder layers: 2
Feed-forward dimension: 512
Dropout: 0.1

Dimension per attention head: 32


In [5]:
#Creating the embedding layer
import torch.nn as nn

embedding = nn.Embedding(
    num_embeddings=VOCAB_SIZE,
    embedding_dim=D_MODEL,
    padding_idx=PAD_ID
)

print("Embedding layer created successfully.")
print("Embedding matrix shape:", embedding.weight.shape)

Embedding layer created successfully.
Embedding matrix shape: torch.Size([9537, 128])


In [7]:
BATCH_SIZE = 16

source_batch = train_data["source"][:BATCH_SIZE]

print("Source token IDs shape:", source_batch.shape)

Source token IDs shape: torch.Size([16, 512])


In [8]:
#Test source embeddings
source_embeddings = embedding(source_batch)

print("Source token IDs shape:", source_batch.shape)
print("Source embedding shape:", source_embeddings.shape)

print("\nFirst token ID:", source_batch[0, 0].item())
print("First token embedding shape:", source_embeddings[0, 0].shape)

Source token IDs shape: torch.Size([16, 512])
Source embedding shape: torch.Size([16, 512, 128])

First token ID: 5
First token embedding shape: torch.Size([128])


In [9]:
#Create positional encoding
import math

def create_positional_encoding(max_length, d_model):
    """
    Create sinusoidal positional encodings.

    Shape:
        [max_length, d_model]
    """

    position = torch.arange(max_length).unsqueeze(1)

    div_term = torch.exp(
        torch.arange(0, d_model, 2) *
        (-math.log(10000.0) / d_model)
    )

    pe = torch.zeros(max_length, d_model)

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    return pe


print("Positional encoding function created successfully.")

Positional encoding function created successfully.


In [11]:
SOURCE_MAX_LENGTH = train_data["source"].shape[1]

print("Source maximum length:", SOURCE_MAX_LENGTH)

Source maximum length: 512


In [12]:
#Generate positional encodings
positional_encoding = create_positional_encoding(
    max_length=SOURCE_MAX_LENGTH,
    d_model=D_MODEL
)

print("Positional encoding shape:", positional_encoding.shape)

print("\nFirst position:")
print(positional_encoding[0])

print("\nSecond position:")
print(positional_encoding[1])

Positional encoding shape: torch.Size([512, 128])

First position:
tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1.])

Second position:
tensor([8.4147e-01, 5.4030e-01, 7.6172e-01, 6.4791e-01, 6.8156e-01, 7.3176e-01,
        6.0469e-01, 7.9646e-01, 5.3317e-01, 8.4601e-01, 4.6795e-01, 8.8376e-01,
        4.0931e-01, 9.1240e-01, 3.5711e-01, 9.3406e-01, 3.1098e-01, 9.5042e-01,
        2.7043e-01, 9.6274e-01, 2.3492e-01, 9.7201e-01, 2.0391e-01, 9.7899e-01,
        1.7689e-01

In [13]:
#Combine embeddings + positional encoding
source_embeddings_with_position = (
    source_embeddings + positional_encoding
)

print("Source embeddings shape:", source_embeddings.shape)
print("Positional encoding shape:", positional_encoding.shape)
print("Combined representation shape:", source_embeddings_with_position.shape)

Source embeddings shape: torch.Size([16, 512, 128])
Positional encoding shape: torch.Size([512, 128])
Combined representation shape: torch.Size([16, 512, 128])


## Q, K, and V


- Query (Q): What information am I looking for?
- Key (K): What information do I contain that can be matched against a query?
- Value (V): What information should I actually pass forward?

In [15]:
# Recreate the combined representation

source_batch = train_data["source"][:BATCH_SIZE]

source_embeddings = embedding(source_batch)

combined_representation = source_embeddings + positional_encoding

print("Combined representation shape:", combined_representation.shape)

Combined representation shape: torch.Size([16, 512, 128])


In [16]:
# Create the Query, Key, and Value projection layers

query_projection = nn.Linear(D_MODEL, D_MODEL)
key_projection = nn.Linear(D_MODEL, D_MODEL)
value_projection = nn.Linear(D_MODEL, D_MODEL)

# Project the combined representation into Q, K, and V
Q = query_projection(combined_representation)
K = key_projection(combined_representation)
V = value_projection(combined_representation)

print("Combined representation shape:", combined_representation.shape)
print("Query (Q) shape:", Q.shape)
print("Key (K) shape:", K.shape)
print("Value (V) shape:", V.shape)

Combined representation shape: torch.Size([16, 512, 128])
Query (Q) shape: torch.Size([16, 512, 128])
Key (K) shape: torch.Size([16, 512, 128])
Value (V) shape: torch.Size([16, 512, 128])


In [19]:
# Split into attention heads why we are doing this b/c instead of one attention operation over 128 dimensions,
# the Transformer creates 4 separate attention heads, each working with 32 dimensions.
# why 32 dimensions b/c 128/4=32
# which means we will transform [Batch=16, Sequence=512, D_MODEL=128] to [Batch=16, Heads=4, Sequence=512, Head_Dimension=32]
# we call it multihead reshaping.

In [18]:
# Dimension of each attention head
HEAD_DIM = D_MODEL // NUM_HEADS

# Reshape Q, K, and V into multiple attention heads
Q_heads = Q.view(BATCH_SIZE, -1, NUM_HEADS, HEAD_DIM).transpose(1, 2)
K_heads = K.view(BATCH_SIZE, -1, NUM_HEADS, HEAD_DIM).transpose(1, 2)
V_heads = V.view(BATCH_SIZE, -1, NUM_HEADS, HEAD_DIM).transpose(1, 2)

print("Head dimension:", HEAD_DIM)
print("Q heads shape:", Q_heads.shape)
print("K heads shape:", K_heads.shape)
print("V heads shape:", V_heads.shape)

Head dimension: 32
Q heads shape: torch.Size([16, 4, 512, 32])
K heads shape: torch.Size([16, 4, 512, 32])
V heads shape: torch.Size([16, 4, 512, 32])


In [22]:
# Now we can see Query [16,512,128] split to query head [16,4,512,32] same with key and value.
# Now we will calculate attention scores which means how much attention to give to each word.
# How relevant is every other token to me
# To make matrix 512 * 512 we have to transpose K's last two dimensions
# Why we are transposing last two dimensions of K's b/c
# For each of the 512 tokens, calculate its attention score against all 512 tokens 
# Each number is a raw attention score produced by QK^t

In [21]:
# Calculate raw attention scores

attention_scores = torch.matmul(
    Q_heads,
    K_heads.transpose(-2, -1)
)

print("Q heads shape:", Q_heads.shape)
print("K heads transposed shape:", K_heads.transpose(-2, -1).shape)
print("Attention scores shape:", attention_scores.shape)

Q heads shape: torch.Size([16, 4, 512, 32])
K heads transposed shape: torch.Size([16, 4, 32, 512])
Attention scores shape: torch.Size([16, 4, 512, 512])


In [23]:
# These raw scores can become relatively large because we're taking dot products across 32 dimensions
# That the reason transformer will use underroot dk
# example: d_k 32 and d_k underroot 32 this is called scaled dot-product attention

In [24]:
# Scale the attention scores

scale = HEAD_DIM ** 0.5

scaled_attention_scores = attention_scores / scale

print("Head dimension:", HEAD_DIM)
print("Scaling factor:", scale)
print("Scaled attention scores shape:", scaled_attention_scores.shape)

Head dimension: 32
Scaling factor: 5.656854249492381
Scaled attention scores shape: torch.Size([16, 4, 512, 512])


In [25]:
# Create a mask for padding tokens

source_padding_mask = source_batch == PAD_ID

print("Source batch shape:", source_batch.shape)
print("Padding mask shape:", source_padding_mask.shape)
print("Number of padding tokens:", source_padding_mask.sum().item())

Source batch shape: torch.Size([16, 512])
Padding mask shape: torch.Size([16, 512])
Number of padding tokens: 3763


In [26]:
# Reshape padding mask for attention broadcasting

attention_padding_mask = source_padding_mask[:, None, None, :]

print("Original padding mask shape:", source_padding_mask.shape)
print("Attention padding mask shape:", attention_padding_mask.shape)

Original padding mask shape: torch.Size([16, 512])
Attention padding mask shape: torch.Size([16, 1, 1, 512])


In [27]:
# Mask padding positions in the attention scores

masked_attention_scores = scaled_attention_scores.masked_fill(
    attention_padding_mask,
    float("-inf")
)

print("Masked attention scores shape:", masked_attention_scores.shape)

Masked attention scores shape: torch.Size([16, 4, 512, 512])


In [28]:
# Convert attention scores into attention weights

attention_weights = torch.softmax(
    masked_attention_scores,
    dim=-1
)

print("Attention weights shape:", attention_weights.shape)

# Check that probabilities sum to 1
attention_sum = attention_weights.sum(dim=-1)

print("Minimum row sum:", attention_sum.min().item())
print("Maximum row sum:", attention_sum.max().item())

Attention weights shape: torch.Size([16, 4, 512, 512])
Minimum row sum: 0.9999997019767761
Maximum row sum: 1.000000238418579


In [29]:
# Check attention weights assigned to padding positions

padding_attention_weights = attention_weights.masked_select(
    attention_padding_mask.expand_as(attention_weights)
)

print("Number of padding attention values:", padding_attention_weights.numel())
print("Maximum attention given to padding:", padding_attention_weights.max().item())

Number of padding attention values: 7706624
Maximum attention given to padding: 0.0


In [30]:
# Compute the weighted combination of Value vectors

attention_output_heads = torch.matmul(
    attention_weights,
    V_heads
)

print("Attention weights shape:", attention_weights.shape)
print("V heads shape:", V_heads.shape)
print("Attention output per head shape:", attention_output_heads.shape)

Attention weights shape: torch.Size([16, 4, 512, 512])
V heads shape: torch.Size([16, 4, 512, 32])
Attention output per head shape: torch.Size([16, 4, 512, 32])


In [31]:
# Combine the outputs from all attention heads

attention_output_combined = attention_output_heads.transpose(1, 2).contiguous()

attention_output_combined = attention_output_combined.view(
    BATCH_SIZE,
    -1,
    D_MODEL
)

print("Attention output per head:", attention_output_heads.shape)
print("Combined attention output:", attention_output_combined.shape)

Attention output per head: torch.Size([16, 4, 512, 32])
Combined attention output: torch.Size([16, 512, 128])


In [32]:
import torch.nn as nn

# Final linear projection after combining all attention heads

output_projection = nn.Linear(D_MODEL, D_MODEL)

attention_output = output_projection(
    attention_output_combined
)

print("Combined attention output shape:", attention_output_combined.shape)
print("Final attention output shape:", attention_output.shape)

Combined attention output shape: torch.Size([16, 512, 128])
Final attention output shape: torch.Size([16, 512, 128])
